# 02 — Extension Tables

**Steps:**
1. Load core stop event table
2. Load and clean DWD weather data → weather extension table
3. Join weather to core table → new combined table
4. Load vehicle information table
5. Join vehicle info to core table → new combined table

## 0. Imports and paths

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import polars as pl

sys.path.append("..")

from pipeline.extensions import (
    load_weather, join_weather,
    load_vehicle_info, join_vehicle_info,
)

In [3]:
CORE_TABLE_PATH    = "../data/processed/core_stop_events.parquet"
WEATHER_RAW_PATH   = "../data/external/weather/produkt_klima_tag_20241129_20260601_01048.txt"
WEATHER_OUT_PATH   = "../data/external/weather/weather_dresden.parquet"
VEHICLE_RAW_PATH   = "../data/vehicle_data_2025_07_22.csv"
VEHICLE_LOOKUP_PATH = "../data/external/vehicle/dvb_fahrzeug_info.csv"
VEHICLE_OUT_PATH   = "../data/external/vehicle/vehicle_info.parquet"
JOINED_WEATHER_PATH  = "../data/processed/core_stop_events_with_weather.parquet"
JOINED_VEHICLE_PATH  = "../data/processed/core_stop_events_with_vehicle.parquet"

## 1. Load core stop event table

In [4]:
core = pl.read_parquet(CORE_TABLE_PATH)
print(f"Core table rows: {len(core):,}")
print(f"Date range: {core['arrival_time'].min()} → {core['arrival_time'].max()}")
core.head(3)

Core table rows: 994,117
Date range: 2025-07-28 00:00:04.519711+00:00 → 2025-08-03 23:59:47.141173+00:00


fzg_id,drop_row_idx,arrival_time,departure_time,linie,fahrt_id,ort_nr_start,stop_index,stop_status,scheduled_arrival_time,delay_calculated_sec,delay_recorded_sec,dwell_time,travel_time,besetztgrad,is_peak_hour,is_workday,has_traffic_signal
i64,i64,"datetime[μs, UTC]","datetime[μs, UTC]",i64,i64,i64,i64,str,"datetime[μs, UTC]",f64,i64,f64,f64,i64,i8,i8,i8
151,106,2025-07-29 02:23:54.604010 UTC,2025-07-29 02:23:54.604010 UTC,4,5924129,184104,0,"""normal""",2025-07-29 02:22:00 UTC,114.60401,-10,0.0,null,1,0,1,null
151,116,2025-07-29 02:25:44.604703 UTC,2025-07-29 02:25:44.604703 UTC,4,5924129,182302,1,"""no_door""",2025-07-29 02:24:00 UTC,104.604703,-20,0.0,110.000693,0,0,1,null
151,124,2025-07-29 02:26:37.622760 UTC,2025-07-29 02:26:37.622760 UTC,4,5924129,182402,2,"""no_door""",2025-07-29 02:26:00 UTC,37.62276,-20,0.0,53.018057,0,0,1,null


## 2. Load and clean DWD weather data

In [8]:
weather = load_weather(WEATHER_RAW_PATH)

print(f"Weather table rows: {len(weather):,}")
print(f"Date range: {weather['MESS_DATUM'].min()} → {weather['MESS_DATUM'].max()}")
weather.head(3)

Weather table rows: 550
Date range: 2024-11-29 → 2026-06-01


MESS_DATUM,TMK,TXK,TNK,TGK,RSK,RSKF,SDK,SHK_TAG,FM
date,f64,f64,f64,f64,f64,f64,f64,f64,f64
2024-11-29,3.3,4.8,-0.1,-3.1,0.0,6.0,0.2,0.0,4.3
2024-11-30,0.9,6.2,-4.0,-4.6,0.0,0.0,7.0,0.0,2.8
2024-12-01,0.5,4.8,-1.4,-2.8,0.0,4.0,7.7,0.0,4.7


In [9]:
# Null check
print("Null counts per column:")
for col in weather.columns:
    n = weather[col].null_count()
    print(f"  {col:<25} {n}")

Null counts per column:
  MESS_DATUM                0
  TMK                       0
  TXK                       0
  TNK                       0
  TGK                       0
  RSK                       0
  RSKF                      0
  SDK                       0
  SHK_TAG                   0
  FM                        0


In [10]:
weather.write_parquet(WEATHER_OUT_PATH)
print(f"Weather table saved to {WEATHER_OUT_PATH}")

Weather table saved to ../data/external/weather/weather_dresden.parquet


## 3. Join weather to core table

In [14]:
core_with_weather = join_weather(core, weather)

total = len(core_with_weather)
nulls = core_with_weather["TMK"].null_count()
print(f"Rows: {total:,}")
print(f"With weather data:    {total-nulls:,}  ({(total-nulls)/total*100:.1f}%)")
print(f"Without weather data: {nulls:,}  ({nulls/total*100:.1f}%)")

core_with_weather.write_parquet(JOINED_WEATHER_PATH)
print(f"Saved to {JOINED_WEATHER_PATH}")

Rows: 994,117
With weather data:    994,117  (100.0%)
Without weather data: 0  (0.0%)
Saved to ../data/processed/core_stop_events_with_weather.parquet


## 4. Load vehicle information table

In [5]:
vehicle_info = load_vehicle_info(VEHICLE_RAW_PATH, VEHICLE_LOOKUP_PATH)

total_v   = len(vehicle_info)
matched_v = vehicle_info["fahrzueg_type"].drop_nulls().len()
print(f"Total vehicles:   {total_v:,}")
print(f"Matched to lookup: {matched_v:,}  ({matched_v/total_v*100:.1f}%)")
print(f"Unmatched:         {total_v-matched_v:,}  ({(total_v-matched_v)/total_v*100:.1f}%)")
print()
print("fahrzueg_type distribution:")
print(vehicle_info.group_by("fahrzueg_type").len().sort("len", descending=True))
vehicle_info.head(5)

Total vehicles:   470
Matched to lookup: 343  (73.0%)
Unmatched:         127  (27.0%)

fahrzueg_type distribution:
shape: (3, 2)
┌───────────────┬─────┐
│ fahrzueg_type ┆ len │
│ ---           ┆ --- │
│ str           ┆ u32 │
╞═══════════════╪═════╡
│ tram          ┆ 189 │
│ bus           ┆ 154 │
│ null          ┆ 127 │
└───────────────┴─────┘


fzg_id,fzg_nr,fzg_nr_int,typ,fahrzueg_type,fahrgasttüren,länge_m,sitzplätze,stehplätze,kapazitaet
f64,str,i64,str,str,i64,f64,i64,i64,i64
6.0,"""201 006-1""",201006,"""SchiebeFz.""",null,null,null,null,null,null
40.0,"""201 122-2""",201122,"""FL-Meßwg.""",null,null,null,null,null,null
41.0,"""201 001-2""",201001,"""Schleifwg.""",null,null,null,null,null,null
81.0,"""201 005-3""",201005,"""SchiebeFz.""",null,null,null,null,null,null
119.0,"""201 317-7""",201317,"""T4D MT""",null,null,null,null,null,null


In [17]:
# Check unmatched vehicles
unmatched = vehicle_info.filter(pl.col("fahrzueg_type").is_null())
if len(unmatched) > 0:
    print(f"Unmatched vehicles ({len(unmatched)}):")
    print(unmatched.select(["fzg_id", "fzg_nr", "fzg_nr_int", "typ"]))
else:
    print("All vehicles matched.")

Unmatched vehicles (127):
shape: (127, 4)
┌────────┬───────────┬────────────┬────────────┐
│ fzg_id ┆ fzg_nr    ┆ fzg_nr_int ┆ typ        │
│ ---    ┆ ---       ┆ ---        ┆ ---        │
│ f64    ┆ str       ┆ i64        ┆ str        │
╞════════╪═══════════╪════════════╪════════════╡
│ 6.0    ┆ 201 006-1 ┆ 201006     ┆ SchiebeFz. │
│ 40.0   ┆ 201 122-2 ┆ 201122     ┆ FL-Meßwg.  │
│ 41.0   ┆ 201 001-2 ┆ 201001     ┆ Schleifwg. │
│ 81.0   ┆ 201 005-3 ┆ 201005     ┆ SchiebeFz. │
│ 119.0  ┆ 201 317-7 ┆ 201317     ┆ T4D MT     │
│ …      ┆ …         ┆ …          ┆ …          │
│ 2162.0 ┆ 900 809-3 ┆ 900809     ┆ MB O530 C2 │
│ 2163.0 ┆ 900 810-8 ┆ 900810     ┆ MB O530 C2 │
│ 2164.0 ┆ 900 407-3 ┆ 900407     ┆ MB O530KC2 │
│ 2165.0 ┆ 900 406-5 ┆ 900406     ┆ MB O530KC2 │
│ 4095.0 ┆ 459 990-2 ┆ 459990     ┆ MBReisebus │
└────────┴───────────┴────────────┴────────────┘


In [18]:
vehicle_info.write_parquet(VEHICLE_OUT_PATH)
print(f"Vehicle info table saved to {VEHICLE_OUT_PATH}")

Vehicle info table saved to ../data/external/vehicle/vehicle_info.parquet


## 5. Join vehicle info to core table

In [20]:
core_with_vehicle = join_vehicle_info(core, vehicle_info)

total = len(core_with_vehicle)
nulls = core_with_vehicle["fahrzueg_type"].null_count()
print(f"Rows: {total:,}")
print(f"With vehicle info:    {total-nulls:,}  ({(total-nulls)/total*100:.1f}%)")
print(f"Without vehicle info: {nulls:,}  ({nulls/total*100:.1f}%)")

core_with_vehicle.write_parquet(JOINED_VEHICLE_PATH)
print(f"Saved to {JOINED_VEHICLE_PATH}")
core_with_vehicle.head(3)

Rows: 994,117
With vehicle info:    856,718  (86.2%)
Without vehicle info: 137,399  (13.8%)
Saved to ../data/processed/core_stop_events_with_vehicle.parquet


fzg_id,drop_row_idx,arrival_time,departure_time,linie,fahrt_id,ort_nr_start,stop_index,stop_status,scheduled_arrival_time,delay_calculated_sec,delay_recorded_sec,dwell_time,travel_time,besetztgrad,is_peak_hour,is_workday,has_traffic_signal,fahrzueg_type,fahrgasttüren,länge_m,sitzplätze,stehplätze,kapazitaet
i64,i64,"datetime[μs, UTC]","datetime[μs, UTC]",i64,i64,i64,i64,str,"datetime[μs, UTC]",f64,i64,f64,f64,i64,i8,i8,i8,str,i64,f64,i64,i64,i64
151,106,2025-07-29 02:23:54.604010 UTC,2025-07-29 02:23:54.604010 UTC,4,5924129,184104,0,"""normal""",2025-07-29 02:22:00 UTC,114.60401,-10,0.0,null,1,0,1,null,"""tram""",6,43.29,97,193,290
151,116,2025-07-29 02:25:44.604703 UTC,2025-07-29 02:25:44.604703 UTC,4,5924129,182302,1,"""no_door""",2025-07-29 02:24:00 UTC,104.604703,-20,0.0,110.000693,0,0,1,null,"""tram""",6,43.29,97,193,290
151,124,2025-07-29 02:26:37.622760 UTC,2025-07-29 02:26:37.622760 UTC,4,5924129,182402,2,"""no_door""",2025-07-29 02:26:00 UTC,37.62276,-20,0.0,53.018057,0,0,1,null,"""tram""",6,43.29,97,193,290


In [6]:
from pipeline.extensions import load_special_events, join_special_events

events = load_special_events("../data/external/events/special_events.csv")
print(events)

core_with_events = join_special_events(core, events)
print(f"Stop events on special event days: {core_with_events['has_special_event'].sum():,}")

shape: (4, 6)
┌──────────┬────────────┬────────────┬─────────────────────────────────┬───────────┬───────────┐
│ event_id ┆ date       ┆ event_type ┆ location                        ┆ latitude  ┆ longitude │
│ ---      ┆ ---        ┆ ---        ┆ ---                             ┆ ---       ┆ ---       │
│ str      ┆ date       ┆ str        ┆ str                             ┆ f64       ┆ f64       │
╞══════════╪════════════╪════════════╪═════════════════════════════════╪═══════════╪═══════════╡
│ E001     ┆ 2025-08-09 ┆ sports     ┆ Rudolf-Harbig-Stadion           ┆ 51.040855 ┆ 13.748034 │
│ E002     ┆ 2025-08-13 ┆ sports     ┆ AOK PLUS Walter-Fritzsch-Akade… ┆ 51.073952 ┆ 13.713271 │
│ E003     ┆ 2025-08-18 ┆ sports     ┆ Rudolf-Harbig-Stadion           ┆ 51.040855 ┆ 13.748034 │
│ E004     ┆ 2025-08-31 ┆ sports     ┆ Rudolf-Harbig-Stadion           ┆ 51.040855 ┆ 13.748034 │
└──────────┴────────────┴────────────┴─────────────────────────────────┴───────────┴───────────┘
Stop events on s